### Comparing to Mattergen

- Convert the dataset used in the pre-train benefits study to a format that can be ussed by mattergen
- Preserves the splits etc

In [ ]:
import __init__
import datasets
from _utils import process_dataframe, filter_df_to_context

# load the dataset c-bone/mpdb-2prop_clean
dataset = datasets.load_dataset("c-bone/mpdb-2prop_clean")

# load as a dataframe
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

dfs = {
    "df_train": df_train,
    "df_val": df_val
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

    print(f"length before filtering: {len(df)}")
    df = filter_df_to_context(df=df, context=1024, num_workers=64)
    print(f"after filtering: {len(df)}")

    df_clean = process_dataframe(df=df, num_workers=64, column_name="CIF")

    df_clean = df_clean.rename(columns={
        'CIF': 'cif',
        'Database': 'material_id',
        'Bandgap (eV)': 'dft_band_gap',
        'Energy Above Hull (eV)': 'energy_above_hull'
    })
    df_clean['material_id'] = [f"id_{i}" for i in range(len(df_clean))]
    df_clean = df_clean[["material_id", "cif", "dft_band_gap", "energy_above_hull"]]
    df_clean.to_csv(f'/home/cyprien/mattergen/datasets/mp_bandgap/{name}.csv', index=False)

### Training

- Mattergen repo was cloned, the new bandgap dataset was added to the repo
- Then made some custom non-default configs which can be found in `_config_files/training/conditional/pretraining_benefits/mattergen_configs`
- All the other configs stayed the same

> Note: There's a few differences compared to our pipeline to consider during comparison
> 1. The model trains on Reduced Niggli Structures whereas we train on Conventional CIFs
> 2. We train from scratch with CIF + Condition pairs whereas Mattergen has to do a base unconditional training, and then we need to do fine-tuning with adapter modules to allow for conditional generation
> 3. We take mattergen's mp-20 configuration and adapt it for this dataset because it is the most similar config from their repository available

> For point 3, this means:
> - Mattergen unconditional is trained for 900 epochs, our from scratch model is trained on around 23 epochs (with early stopping)
> - They use a batch size of 512, whereas ours is 32
> - They use ReduceLROnPlateau Schedule, whereas we use a CosineDecayWithMinLR
> - Training time is vastly different. 900 epochs is around 80h on 3x40GB GPUs
> - vs 23 epochs around 2h30m on 2x24GB GPUs

During generation, mattergen sets the distribution of N_ATOMS during generation to the one seen in training. So, we need to change this dsitribution in the code to the one in our dataset. Its calculated as below. 

In [2]:
import __init__

Navigated to package root


In [ ]:
import __init__
import pandas as pd
import _utils._notebook_utils as notebook_utils

compute_atom_counts = notebook_utils.compute_atom_counts
load_count_datasets = notebook_utils.load_count_datasets

loaded = load_count_datasets([("MP Bandgap", "HF-databases/mpdb_processing/mpdb_2prop_count.parquet", 1024)])

loaded = compute_atom_counts(loaded)

def compute_atom_count_distribution(df):
    count_series = df['prim_count']
    value_counts = count_series.value_counts(normalize=True).sort_index()
    return value_counts.to_dict()

atom_count_distributions = compute_atom_count_distribution(loaded[0][1])

atom_count_distributions

In [8]:
df = pd.read_parquet('_artifacts/pretrain_benefits/scratch-methods/mpdb-scratch-PKV_gen.parquet')
df

,Material ID,Prompt,Generated CIF,Condition Vector
0,Generated_1,<bos>\ndata_[,\ndata_La20Sb16\nloop_\n _atom_type_symbol\n _...,"0.05, 0.0"
1,Generated_2,<bos>\ndata_[,\ndata_In24As16\nloop_\n _atom_type_symbol\n _...,"0.05, 0.0"
2,Generated_3,<bos>\ndata_[,\ndata_Cs4V4Se8O24\nloop_\n _atom_type_symbol\...,"0.05, 0.0"
3,Generated_4,<bos>\ndata_[,\ndata_Eu12Al8As16\nloop_\n _atom_type_symbol\...,"0.05, 0.0"
4,Generated_5,<bos>\ndata_[,\ndata_La4Sb4Pd4\nloop_\n _atom_type_symbol\n ...,"0.05, 0.0"
...,...,...,...,...
7995,Generated_3996,<bos>\ndata_[,\ndata_Ba4B4F16\nloop_\n _atom_type_symbol\n _...,"0.9, 0.0"
7996,Generated_3997,<bos>\ndata_[,\ndata_Ba4.Y20F80\nloop_\n _atom_type_symbol\n...,"0.9, 0.0"
7997,Generated_3998,<bos>\ndata_[,\ndata_Ba2Tm2F12\nloop_\n _atom_type_symbol\n ...,"0.9, 0.0"
7998,Generated_3999,<bos>\ndata_[,\ndata_He8\nloop_\n _atom_type_symbol\n _atom_...,"0.9, 0.0"


In [ ]:
!python _utils/_metrics/VUN_metrics.py \
    --input_parquet '/home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif.parquet' \
    --huggingface_dataset 'c-bone/mpdb-2prop_clean' \
    --load_processed_data 'HF-databases/mpdb-2prop_clean/mpdb_2prop_proc.parquet' \
    --output_parquet '/home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif_metrics.parquet' \
    --num_workers 32 \
    --allow_stated_p1_mismatch

In [1]:
!python _utils/_metrics/mace_ehull.py \
    --post_parquet /home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif_metrics.parquet \
    --output_parquet /home/cyprien/mattergen/gen_tests/test3_L_uncond/generated_crystals_cif_metrics.parquet \
    --mp_data 'mp_computed_structure_entries.json.gz' \
    --num_workers 16

python: can't open file '/home/cyprien/CrystaLLM-pi-paper-v2/notebooks/_utils/_metrics/mace_ehull.py': [Errno 2] No such file or directory
